<a href="https://colab.research.google.com/github/wjtopp3/CSIT-2033/blob/main/CSIT_Week_13_Lab_1_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fix the vulnerable code.

**Background:**

You are a security analyst asked to review a developer's Python code. The program is supposed to accept a JSON file and display a file from a "reports" directory.

**The Problem:**

The code you are given is exactly 10 lines. The developer claims it accepts a purported JSON request and displays a file from a "reports" directory as requested. You find that the developer did not follow Python or OpenSSF standards and you determine the code is vulnerable.

**Your task:**
- ***Identify*** the unsafe parsing, missing input validation, unsafe file-path handling, and resource/error-handling problems
- ***Write an executive summary*** outlining and explaining your findings with supporting documentation (i.e., references and citations)
- ***Rewrite the program*** so that it accepts only a JSON object containing an approved report name.

**Submit:**
- **This .ipynb notebook** with the original code and the tested, corrected code.
- **An executive summary** that you would submit to the development manager to show your findings. The length of the document is at your discretion.

# **Here is the developer's code snippet.**

In [ ]:
from pathlib import Path
base = Path("reports")
raw = input("JSON request: ")
request = eval(raw)
filename = request.get("file", "summary.txt")
path = base / filename
handle = open(path, "r", encoding="utf-8")
contents = handle.read()
handle.close()
print(contents)

# Rewrite the developer's code in secure, OpenSSF-standard, non-vulnerable Python code.

In [ ]:
# Enter your code below this line

# **3. Rewritten version**

The corrected version is deliberately longer than 10 lines so that validation and error handling remain readable. Place approved UTF-8 reports in a reports directory alongside the script.

In [ ]:
import json
from pathlib import Path

REPORT_DIR = Path(__file__).resolve().parent / "reports"
ALLOWED_REPORTS = {"summary.txt", "attendance.txt"}
MAX_REQUEST_CHARS = 2048
MAX_REPORT_CHARS = 100_000


def main():
    try:
        raw = input("JSON request: ")
        if len(raw) > MAX_REQUEST_CHARS:
            raise ValueError("Request too large")

        request = json.loads(raw)
        if not isinstance(request, dict) or set(request) != {"file"}:
            raise ValueError("Expected a file field")

        filename = request["file"]
        if not isinstance(filename, str) or filename not in ALLOWED_REPORTS:
            raise ValueError("Report not allowed")

        base = REPORT_DIR.resolve(strict=True)
        path = (base / filename).resolve(strict=True)
        if path.parent != base or not path.is_file():
            raise ValueError("Invalid report path")

        with path.open("r", encoding="utf-8") as handle:
            contents = handle.read(MAX_REPORT_CHARS + 1)

        if len(contents) > MAX_REPORT_CHARS:
            raise ValueError("Report too large")

    except (ValueError, OSError, UnicodeError, RuntimeError, EOFError):
        print("Unable to process the request or read an approved report.")
        return

    print(contents)


if __name__ == "__main__":
    main()

JSON input file

{"file": "summary.txt"}

The rewrite uses an explicit allowlist and checks the resolved file location, reflecting OpenSSF’s input-validation and path-canonicalization guidance. Its generic failure message avoids disclosing filesystem details.

**Security assumptions and limitations**

This is a classroom example aligned with OpenSSF guidance, not a certification of complete security. The script and report directory must be administrator-controlled and not writable by untrusted users. Resolving a path and subsequently opening it does not eliminate a filesystem race if an attacker can change files or symbolic links between those operations. OpenSSF likewise cautions that pathlib alone does not prevent all attacks.

The request-length check also happens after input() reads the line; it limits what the program processes, not the memory used to receive it. A network-facing adaptation should enforce request-size limits at its input boundary. This is consistent with validating untrusted input before accepting it for processing.